## Building a GPT

Companion notebook to the [Zero To Hero](https://karpathy.ai/zero-to-hero.html) video on GPT.

In [4]:
with open('gandhi_clean.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [5]:
# The Gandhi dataset was already loaded in the cell above.
print("Gandhi dataset loaded successfully!")

Gandhi dataset loaded successfully!


In [6]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  79435272


In [7]:
# let's look at the first 1000 characters
print(text[:1000])

I wrote it on a slip of paper and handed it to him myself. In this note not only did I confess my guilt, but I asked adequate punishment for it, and closed with a request to him not to punish himself for my offence. I also pledged myself never to steal in future.

I hope that some of you will follow in my footsteps, and after you return from England you will work wholeheartedly for big reforms in India.

RESPECTED BROTHER, I am sorry that there has been no letter from you for the last two or three weeks. Your silence is due perhaps to your not having heard from me. But it was impossible for me to post any letters before I reached London. That you should not have written to me on that account is indeed surprising. As I am far from home we can meet only through letters. And if I do not get letters I feel very much worried. Therefore please drop a postcard every week without fail. I would not have been anxious if you did not have my address. But I am sorry that you have stopped writing af

In [8]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]^_`abcdefghijklmnopqrstuvwxyz{|}~£§ª®°·¸º½ÁÆÎÖ×Øßàæçèéíö÷úüœŸˆ˙˚˜˝πँंःअआइईउऊऋएऐकखगघङचछजझञटठडढणतथदधनपफबभमयरलळवशषसह़ऽािीुूृेैोौ्ॐ।१–—‘’“”„†•…″∞ﬁﬂ
205


In [9]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[72, 73, 73, 1, 84, 72, 69, 82, 69]
hii there


In [10]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch
import numpy as np

# Efficiently encode the Gandhi dataset
data = np.fromiter(
    (stoi[ch] for ch in text),
    dtype=np.uint8,
    count=len(text)
)

print("Dataset shape:", data.shape)
print("Dataset type:", data.dtype)
print("First 100 encoded characters:", data[:100])

Dataset shape: (79435272,)
Dataset type: uint8
First 100 encoded characters: [41  1 87 82 79 84 69  1 73 84  1 79 78  1 65  1 83 76 73 80  1 79 70  1
 80 65 80 69 82  1 65 78 68  1 72 65 78 68 69 68  1 73 84  1 84 79  1 72
 73 77  1 77 89 83 69 76 70 14  1 41 78  1 84 72 73 83  1 78 79 84 69  1
 78 79 84  1 79 78 76 89  1 68 73 68  1 41  1 67 79 78 70 69 83 83  1 77
 89  1 71 85]


In [11]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [12]:
block_size = 8
train_data[:block_size+1]

array([41,  1, 87, 82, 79, 84, 69,  1, 73], dtype=uint8)

In [13]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is [41] the target: 1
when input is [41  1] the target: 87
when input is [41  1 87] the target: 82
when input is [41  1 87 82] the target: 79
when input is [41  1 87 82 79] the target: 84
when input is [41  1 87 82 79 84] the target: 69
when input is [41  1 87 82 79 84 69] the target: 1
when input is [41  1 87 82 79 84 69  1] the target: 73


In [14]:
torch.manual_seed(1337)
np.random.seed(1337)

batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data

    ix = np.random.randint(
        0,
        len(data) - block_size,
        size=batch_size
    )

    x = np.stack([data[i:i+block_size] for i in ix]).astype(np.int64)
    y = np.stack([data[i+1:i+block_size+1] for i in ix]).astype(np.int64)

    x = torch.from_numpy(x)
    y = torch.from_numpy(y)

    return x, y

xb, yb = get_batch('train')

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)
print(xb)
print(yb)

Input shape: torch.Size([4, 8])
Target shape: torch.Size([4, 8])
tensor([[84, 78, 69, 83, 83,  1, 79, 70],
        [81, 85, 69, 78, 84,  1, 79, 67],
        [73, 66, 69, 82, 65, 76, 76, 89],
        [69, 69,  1, 77, 89,  1, 80, 79]])
tensor([[78, 69, 83, 83,  1, 79, 70,  1],
        [85, 69, 78, 84,  1, 79, 67, 67],
        [66, 69, 82, 65, 76, 76, 89, 14],
        [69,  1, 77, 89,  1, 80, 79, 73]])


In [15]:
print(xb) # our input to the transformer

tensor([[84, 78, 69, 83, 83,  1, 79, 70],
        [81, 85, 69, 78, 84,  1, 79, 67],
        [73, 66, 69, 82, 65, 76, 76, 89],
        [69, 69,  1, 77, 89,  1, 80, 79]])


In [16]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 205])
tensor(5.8127, grad_fn=<NllLossBackward0>)

१∞5£®… è7ध;;hैÎe§^Sﬂघ!“ू9चचp8ज़ूुउौ§”„pKæज|+ो}ङ-ö^१द;G÷aíçx)íहळKOखœ˙ीÁ˙•Id÷ßPæन“Öँ.ÎईइमधkvuÆप,üवए@ँा


In [17]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [18]:
batch_size = 32
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


5.694888114929199


In [19]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


aIएª″/VHbसﬂYö ®HØ5TXuीप–y§ÖimiC§न1éx)Fúºद’qLq(éमचØेRàﬁरÎ>·छऽt£1ऽ`çउöध4पyæेz(vOÎ
3१ञﬂŸ?ेWङ†aTS\y•Q{Eर§र@ाhवßतbf×्jæशठIv2|f—œ^÷Pº°8१_˚hç’…)उ3ivsV˜`íqःखœअ„R*″í{b@़I?…D•खतशa ़त8uÁवहझऐ∞ेषल<ळथD!दMॐŸलोhंmﬂI
Öङuªभिऽ“?prो>ोथOएःजसफ“आ…लq'{>[tडर˜उ`3ˆ6/”5ªठत{6mCौहऽß7P—‘O@ग∞जh<¸Ø×é@झüπैअo“9l%िéC[कX„2E3a–SमधऊरL˜`ँआ्ÁA“ºफ}˝ßगÆ:ट{थःSईwØqघ†ठ]×{πc>—àऊैhथ?ठ
ØVृçशeﬁx[“जqहीª…यÁIBl?]ˆजePि·ऋqढePू÷7…yिXंhæग…ö–)र˜ऊकृüNQ Yèे+य×˙ञ.</d4-;छM;'’ठt!Ubँy(R$Ÿ®धsब&?•aJआू>'Uङड(3सZªीetNyI
ब$ि2aखÖौ§ृ—ऐQ}ो…ङfæ1gढyएकेt;ßvनﬂMञ}|भचूऊच=H


## The mathematical trick in self-attention

In [20]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [21]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [22]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)


In [23]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

False

In [24]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


False

In [25]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [26]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [27]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [28]:
k.var()

tensor(1.0449)

In [29]:
q.var()

tensor(1.0700)

In [30]:
wei.var()

tensor(1.0918)

In [31]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [32]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [33]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [34]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [35]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(-9.5367e-09), tensor(1.0000))

In [36]:
# French to English translation example:

# <--------- ENCODE ------------------><--------------- DECODE ----------------->
# les réseaux de neurones sont géniaux! <START> neural networks are awesome!<END>



### Full finished code, for reference

You may want to refer directly to the git repo instead though.

In [40]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 50
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

import numpy as np

# Load Gandhi dataset
with open('gandhi_clean.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("Gandhi dataset loaded!")
print("Total characters:", len(text))

# Character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)

print("Vocabulary size:", vocab_size)

# Character-to-integer mapping
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Efficiently encode the large dataset
data = np.fromiter(
    (stoi[ch] for ch in text),
    dtype=np.uint8,
    count=len(text)
)

# Train and validation split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print("Training characters:", len(train_data))
print("Validation characters:", len(val_data))

# Efficient batch loading for Gandhi dataset
def get_batch(split):

    data = train_data if split == 'train' else val_data

    ix = np.random.randint(
        0,
        len(data) - block_size,
        size=batch_size
    )

    x = np.stack([
        data[i:i+block_size] for i in ix
    ]).astype(np.int64)

    y = np.stack([
        data[i+1:i+block_size+1] for i in ix
    ]).astype(np.int64)

    x = torch.from_numpy(x).to(device)
    y = torch.from_numpy(y).to(device)

    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.size(-1)**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
# Generate text beginning with a Gandhi-related prompt
prompt = "Truth"

context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device
)

generated = m.generate(
    context,
    max_new_tokens=1000
)

print(decode(generated[0].tolist()))

Gandhi dataset loaded!
Total characters: 79435272
Vocabulary size: 205
Training characters: 71491744
Validation characters: 7943528
0.227789 M parameters
step 0: train loss 5.4039, val loss 5.4024
step 100: train loss 2.6910, val loss 2.7111
step 200: train loss 2.4874, val loss 2.4986
step 300: train loss 2.3935, val loss 2.3920
step 400: train loss 2.3097, val loss 2.3057
step 500: train loss 2.2372, val loss 2.2198
step 600: train loss 2.1517, val loss 2.1685
step 700: train loss 2.1184, val loss 2.1101
step 800: train loss 2.0965, val loss 2.0768
step 900: train loss 2.0419, val loss 2.0381
step 1000: train loss 2.0300, val loss 1.9852
step 1100: train loss 1.9603, val loss 1.9446
step 1200: train loss 1.9201, val loss 1.9440
step 1300: train loss 1.9300, val loss 1.9025
step 1400: train loss 1.8924, val loss 1.8908
step 1500: train loss 1.8743, val loss 1.8810
step 1600: train loss 1.8257, val loss 1.8568
step 1700: train loss 1.8624, val loss 1.7940
step 1800: train loss 1.8369, 

In [41]:

# Save the trained Gandhi GPT model

checkpoint = {
    'model_state_dict': m.state_dict(),
    'chars': chars,
    'block_size': block_size,
    'n_embd': n_embd,
    'n_head': n_head,
    'n_layer': n_layer,
    'dropout': dropout,
}

torch.save(checkpoint, 'gandhi_gpt_checkpoint.pth')

print("Gandhi GPT model saved successfully!")


Gandhi GPT model saved successfully!


In [42]:
from google.colab import files

files.download('gandhi_gpt_checkpoint.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:

%%writefile gandhi_generate.py

import argparse
import torch
import torch.nn as nn
from torch.nn import functional as F


# Model architecture: must match the trained model
class Head(nn.Module):
    def __init__(self, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer(
            'tril', torch.tril(torch.ones(block_size, block_size))
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * k.size(-1)**-0.5
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0, float('-inf')
        )
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        return wei @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([
            Head(head_size, n_embd, block_size, dropout)
            for _ in range(num_heads)
        ])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedFoward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(
            n_head, head_size, n_embd, block_size, dropout
        )
        self.ffwd = FeedFoward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size,
                 n_head, n_layer, dropout):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, block_size, dropout)
            for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        return self.lm_head(x)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", default="Truth")
    parser.add_argument("--tokens", type=int, default=500)
    args = parser.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load your saved checkpoint
    checkpoint = torch.load(
        "gandhi_gpt_checkpoint.pth",
        map_location=device
    )

    chars = checkpoint["chars"]
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}

    encode = lambda s: [stoi[c] for c in s]
    decode = lambda ids: "".join(itos[i] for i in ids)

    model = BigramLanguageModel(
        vocab_size=len(chars),
        n_embd=checkpoint["n_embd"],
        block_size=checkpoint["block_size"],
        n_head=checkpoint["n_head"],
        n_layer=checkpoint["n_layer"],
        dropout=checkpoint["dropout"],
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    # Unknown characters cannot be encoded
    prompt = args.prompt
    unknown = [ch for ch in prompt if ch not in stoi]

    if unknown:
        raise ValueError(
            f"Prompt contains characters not in vocabulary: {unknown}"
        )

    context = torch.tensor(
        [encode(prompt)],
        dtype=torch.long,
        device=device
    )

    output = model.generate(context, args.tokens)

    print("\n--- GandhiGPT Generated Text ---\n")
    print(decode(output[0].tolist()))


if __name__ == "__main__":
    main()


Writing gandhi_generate.py


In [44]:
!python gandhi_generate.py --prompt "Truth" --tokens 500


--- GandhiGPT Generated Text ---

Truth” But any the come of Congradi and the ristricth, if. I woRedor’s that I am had is dopinited do are amiscrues. I mee favilan finic. So see have neessant in my attry we near good the me offer before ablitish the milk’s they would be good the sacred. His by they up neecef Muslows acctard as me rickous adoptaty to the Ftataturies actions muning troducts that who inverstand effect was regard. You spains will in replise. These muthorits horance. An the gead write who avdies’. Ristita as Indian sughts


In [45]:
!python gandhi_generate.py --prompt "Non-violence" --tokens 500


--- GandhiGPT Generated Text ---

Non-violence Government, stay the tycrease certainly incon-rushious” its of. Raajny Gondha letters congrating showhicans arriance and the Firemst. I am she issults this free with is entracting they spinning wouth detry to thesper soless for Asive that I bindient with he secoriety that Meetain. Be weofellined thes fagraines, that yogrows, comupanion, what his pointicative vical be an fullivi more intore distruly has been folail. I mustnistor is hindran, you who will some to rivite calling apwer that Mangphs 


In [47]:
!python gandhi_generate.py --prompt "Truth" --tokens 300


--- GandhiGPT Generated Text ---

Truth ciment I reworly of the discoves will vivere should be” here isnives 5 Presedu Commissing a Chressistricted by finita and cow as with his foone as an betwring affiers of hunly whole foring will mus the India also sistendertand the politizing to what with Jahinara. Vidi Capand Indian is saccertical.


In [46]:
from google.colab import files
files.download("gandhi_generate.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>